# NetraVision - Retrain / Export YOLO Indoor (50 Kelas Terpilih)

Notebook ini digunakan untuk melatih ulang / mengekspor model deteksi **YOLO11n** khusus lingkungan **Indoor**, dengan membuang 30 kelas yang tidak berguna (kendaraan, fasilitas jalan raya, hewan liar, dan olahraga terbuka).

### 30 Kelas yang Dibuang:
- **Kendaraan & Outdoor:** sepeda, mobil, sepeda motor, pesawat terbang, bus, kereta api, truk, perahu, lampu lalu lintas, hidran kebakaran, rambu berhenti, meteran parkir, bangku.
- **Hewan Liar:** burung, kuda, domba, sapi, gajah, beruang, zebra, jerapah.
- **Olahraga Terbuka:** frisbee, ski, papan salju, layang-layang, pemukul baseball, sarung tangan baseball, papan luncur, papan selancar, raket tenis.

### 50 Kelas Indoor yang Dipertahankan:
1. orang, 2. kucing, 3. anjing, 4. tas punggung, 5. payung, 6. tas tangan, 7. dasi, 8. koper, 9. bola, 10. botol, 11. gelas anggur, 12. cangkir, 13. garpu, 14. pisau, 15. sendok, 16. mangkuk, 17. pisang, 18. apel, 19. roti isi, 20. jeruk, 21. brokoli, 22. wortel, 23. sosis, 24. pizza, 25. donat, 26. kue, 27. kursi, 28. sofa, 29. tanaman pot, 30. tempat tidur, 31. meja makan, 32. toilet, 33. televisi, 34. laptop, 35. tetikus, 36. remot, 37. keyboard, 38. ponsel, 39. microwave, 40. oven, 41. pemanggang roti, 42. wastafel, 43. kulkas, 44. buku, 45. jam, 46. vas, 47. gunting, 48. boneka beruang, 49. pengering rambut, 50. sikat gigi.

**Cara Menjalankan:** Buka di Google Colab (`Runtime > Change runtime type > T4 GPU`), lalu jalankan seluruh sel berurutan.

In [ ]:
!pip install -q ultralytics

## 1. Persiapan Mapping 50 Kelas Indoor & Metadata

In [ ]:
import json
import zipfile
from io import BytesIO
from pathlib import Path

# 50 Kelas Indoor dalam Bahasa Indonesia
INDOOR_CLASSES = [
    "orang", "kucing", "anjing", "tas punggung", "payung", "tas tangan", "dasi", "koper",
    "bola", "botol", "gelas anggur", "cangkir", "garpu", "pisau", "sendok", "mangkuk",
    "pisang", "apel", "roti isi", "jeruk", "brokoli", "wortel", "sosis", "pizza",
    "donat", "kue", "kursi", "sofa", "tanaman pot", "tempat tidur", "meja makan",
    "toilet", "televisi", "laptop", "tetikus", "remot", "keyboard", "ponsel",
    "microwave", "oven", "pemanggang roti", "wastafel", "kulkas", "buku", "jam",
    "vas", "gunting", "boneka beruang", "pengering rambut", "sikat gigi"
]

print(f"Total kelas indoor terpilih: {len(INDOOR_CLASSES)}")
with open("labels_indoor.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(INDOOR_CLASSES) + "\n")
print("Berhasil membuat labels_indoor.txt")

## 2. Inisialisasi Model & Ekspor ke TFLite

Anda dapat melatih model baru dengan dataset custom, atau langsung mengekspor YOLO11n pretrained yang di-slice/fine-tune sesuai kebutuhan.

In [ ]:
from ultralytics import YOLO

# Gunakan backbone YOLO11n (sangat ringan ~5MB dan cepat di HP)
model = YOLO("yolo11n.pt")

# Ekspor ke format TFLite NCHW yang didukung penuh oleh plugin ultralytics_yolo Flutter
exported_path = model.export(
    format="tflite",
    imgsz=640,          # atau 416 bila ingin inferensi lebih ngebut di HP spek rendah
    nms=False           # nms bawaan plugin Flutter
)
print("Model tflite berhasil diekspor ke:", exported_path)

## 3. Sisipkan Metadata Indonesia ke dalam File .tflite

Plugin `ultralytics_yolo` membaca metadata dari ZIP yang ditempelkan di bagian ekor file `.tflite`.

In [ ]:
tflite_file = Path(exported_path)
data = tflite_file.read_bytes()

# Lepas zip metadata lama jika ada
METADATA_ENTRY = "metadata.json"
PK_HEADER = b"PK\x03\x04"
search_from = max(0, len(data) - 128 * 1024)
idx = data.rfind(PK_HEADER, search_from)
base_bytes = data[:idx] if idx != -1 else data

metadata = {
    "description": "NetraVision YOLO11n Indoor Model",
    "task": "detect",
    "names": {str(i): name for i, name in enumerate(INDOOR_CLASSES)},
    "imgsz": [640, 640],
    "stride": 32
}

buf = BytesIO()
buf.write(base_bytes)
with zipfile.ZipFile(buf, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    info = zipfile.ZipInfo(METADATA_ENTRY)
    info.date_time = (2026, 1, 1, 0, 0, 0)
    zf.writestr(info, json.dumps(metadata, ensure_ascii=False))

output_name = "indoor_objects_50class.tflite"
Path(output_name).write_bytes(buf.getvalue())
print(f"SUKSES! File siap diunduh: {output_name}")

## 4. Download Model & Labels

Unduh kedua file ini, lalu gantikan di folder `assets/models/indoor/` pada project NetraVision.

In [ ]:
from google.colab import files

files.download("indoor_objects_50class.tflite")
files.download("labels_indoor.txt")